# 05 — A taste representation from several seed tracks

**Goal of this step:** turn 3–5 tracks someone likes into a reusable taste profile and a candidate pool. Familiar, bridge and adventurous ranking comes **next**. This notebook does not claim the average alone is a full taste model.

We reuse the *frozen baseline* embedding: FMA small → all 518 audio features → `StandardScaler` → PCA retaining 90% variance → cosine distance. Later we can swap the embedding for `heard` while holding this logic fixed. No listening history is assumed.

Open from the project checkout containing `fma_metadata/features.csv` and `fma_metadata/tracks.csv`. This notebook searches the working directory and its parents for common `data/` locations; it no longer requires `modeling/`. The underlying FMA data was not attached here, so FMA outputs cannot be run in this workspace.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler




# Jupyter's working directory can differ from the notebook's folder.
# Look near the working directory rather than assuming ../data is correct.
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [base / suffix for base in roots for suffix in (
    "data/fma_metadata", "data/raw/fma_metadata", "fma_metadata"
)]
meta_dir = next((d for d in candidates
                 if (d / "features.csv").is_file() and (d / "tracks.csv").is_file()), None)
if meta_dir is None:
    raise FileNotFoundError(
        "FMA metadata files not found. Working directory: " + str(Path.cwd())
        + "\nLooked in:\n" + "\n".join(map(str, candidates))
        + "\nSet meta_dir = Path('/absolute/path/to/fma_metadata') if your data lives elsewhere."
    )
print("FMA metadata:", meta_dir)

tracks = pd.read_csv(meta_dir / "tracks.csv", index_col=0, header=[0, 1])
small_tracks = tracks.loc[tracks[("set", "subset")] == "small"].copy()
features = pd.read_csv(meta_dir / "features.csv", index_col=0, header=[0, 1, 2])
X_small = features.reindex(small_tracks.index).dropna()
small_tracks = small_tracks.loc[X_small.index]
if X_small.empty:
    raise ValueError("No complete FMA-small feature rows matched tracks.csv")

# Exactly the baseline recipe: all 518 features, scale, 90%-variance PCA.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_small)
pca = PCA(n_components=0.90, random_state=42)
X_pca = pd.DataFrame(pca.fit_transform(X_scaled), index=X_small.index)
print(f"{len(X_pca)} tracks, {X_pca.shape[1]} PCA components")



FMA metadata: /Users/tim/neuefische_course/capstone-matchas-tastebridge/data/raw/fma_metadata


8000 tracks, 146 PCA components


## Representation

Cosine compares directions. We first normalize each seed, then take the weighted mean and normalize the result. Equal weights are the default; weights can express *how much* each seed matters. We **retain the individual normalized seed vectors**, because a listener who likes two distinct sounds can have a mean that matches neither. Candidate retrieval uses both the centre and every seed.

The centre is a convenient first summary, **not a claim that taste is one genre or one point**. The table below makes the fit of every seed to the centre visible.

In [2]:
from dataclasses import dataclass


def unit_rows(a):
    a = np.asarray(a, dtype=float)
    norms = np.linalg.norm(a, axis=1, keepdims=True)
    if not np.isfinite(a).all() or (norms == 0).any():
        raise ValueError("Embedding contains non-finite or zero-length rows")
    return a / norms


@dataclass
class TasteProfile:
    seed_ids: tuple
    weights: np.ndarray
    seed_vectors: np.ndarray
    centre: np.ndarray


def build_taste(embedding: pd.DataFrame, seed_ids, weights=None) -> TasteProfile:
    seed_ids = tuple(int(s) for s in seed_ids)
    if len(seed_ids) < 2 or len(set(seed_ids)) != len(seed_ids):
        raise ValueError("Provide at least two distinct seed track IDs")
    missing = [s for s in seed_ids if s not in embedding.index]
    if missing:
        raise KeyError(f"Seed IDs absent from embedding: {missing}")
    w = np.ones(len(seed_ids)) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (len(seed_ids),) or not np.isfinite(w).all() or (w < 0).any() or w.sum() == 0:
        raise ValueError("Weights must be finite, nonnegative, one per seed, with a positive total")
    w = w / w.sum()
    seeds = unit_rows(embedding.loc[list(seed_ids)].to_numpy())
    centre_raw = w @ seeds
    length = np.linalg.norm(centre_raw)
    if length < 1e-10:
        raise ValueError("Seed vectors cancel out; try a different seed set or weights")
    return TasteProfile(seed_ids, w, seeds, centre_raw / length)


def retrieve_candidates(embedding: pd.DataFrame, profile: TasteProfile, per_anchor=100):
    """Union of nearest candidates to the centre AND each seed. No ranking tiers yet."""
    if per_anchor < 1:
        raise ValueError("per_anchor must be positive")
    U = unit_rows(embedding.to_numpy())
    seed_positions = embedding.index.get_indexer(profile.seed_ids)
    if (seed_positions < 0).any():
        raise KeyError("Embedding and taste profile disagree on seed IDs")
    # Recompute seed vectors in this embedding: prevents accidentally mixing feature spaces.
    if not np.allclose(U[seed_positions], profile.seed_vectors, atol=1e-8):
        raise ValueError("Profile was built using a different embedding")
    anchors = np.vstack([profile.centre, profile.seed_vectors])
    index = NearestNeighbors(metric="cosine", algorithm="brute").fit(U)
    n = min(len(U), per_anchor + len(profile.seed_ids) + 1)
    _, positions = index.kneighbors(anchors, n_neighbors=n)
    ids = [int(embedding.index[j]) for j in np.unique(positions)]
    ids = [i for i in ids if i not in profile.seed_ids]
    vectors = U[embedding.index.get_indexer(ids)]
    similarities = vectors @ profile.seed_vectors.T
    result = pd.DataFrame({
        "centre_similarity": vectors @ profile.centre,
        "closest_seed_similarity": similarities.max(axis=1),
        "closest_seed_id": np.asarray(profile.seed_ids)[similarities.argmax(axis=1)],
    }, index=pd.Index(ids, name="track_id"))
    result["centre_distance"] = 1 - result["centre_similarity"]
    result["closest_seed_distance"] = 1 - result["closest_seed_similarity"]
    return result.sort_values("centre_similarity", ascending=False)


## Try a real seed set

These three IDs occur in the existing blind-test seed list. They are merely a reproducible example: replace them with 3–5 tracks representing **one person's actual tastes** before interpreting recommendations. The `per_anchor` setting determines how wide a candidate pool is available for later levels; it does not set adventure by rank.

In [3]:
#SEED_IDS = [140, 54033, 59876]  # Replace with your user's 3–5 liked tracks
SEED_IDS = [704, 1083, 56690]
profile = build_taste(X_pca, SEED_IDS)
candidates = retrieve_candidates(X_pca, profile, per_anchor=150)

seed_meta = small_tracks.loc[SEED_IDS]
seed_view = pd.DataFrame({
    "artist": seed_meta[("artist", "name")].values,
    "title": seed_meta[("track", "title")].values,
    "genre": seed_meta[("track", "genre_top")].values,
    "weight": profile.weights,
    "similarity_to_centre": profile.seed_vectors @ profile.centre,
}, index=pd.Index(SEED_IDS, name="track_id"))
display(seed_view)
print(f"{len(candidates)} distinct candidates, excluding the {len(SEED_IDS)} seeds")
print("Seed-to-seed cosine similarities (low values suggest separate taste regions):")
display(pd.DataFrame(profile.seed_vectors @ profile.seed_vectors.T,
                     index=SEED_IDS, columns=SEED_IDS).round(2))

preview = candidates.head(12).join(pd.DataFrame({
    "artist": small_tracks[("artist", "name")],
    "title": small_tracks[("track", "title")],
    "genre": small_tracks[("track", "genre_top")],
}))
display(preview[["artist", "title", "genre", "centre_similarity",
                 "closest_seed_id", "closest_seed_similarity"]])

,artist,title,genre,weight,similarity_to_centre
track_id,,,,,
704,Fósforo,Mano De Dios,International,0.333333,0.667695
1083,Magic Lantern,Feasting on Energy,Experimental,0.333333,0.442978
56690,Corty,Trod Along (vocals: Preecha Kungo),International,0.333333,0.651796


504 distinct candidates, excluding the 3 seeds
Seed-to-seed cosine similarities (low values suggest separate taste regions):


,704,1083,56690
704,1.00,-0.10,0.27
1083,-0.10,1.00,-0.12
56690,0.27,-0.12,1.00


,artist,title,genre,centre_similarity,closest_seed_id,closest_seed_similarity
track_id,,,,,,
21995,Katapulto,zero rhythm dollar boy,Pop,0.648414,704,0.402456
68891,Broke For Free,Building The Sun,Electronic,0.598683,56690,0.389145
48442,Bass Assassin,8bitDetective,Electronic,0.590306,704,0.430316
87069,Waste Disposal Machine,I Sing the Body Electric (first take SFI remix),Electronic,0.588708,1083,0.417655
55242,Mastermind XS,The Essence Of Bottled Light,International,0.588294,1083,0.549856
36998,Exteenager,bonsoir tiroir!,Electronic,0.582421,704,0.584773
56030,Mastermind XS,All The Stars,International,0.581419,704,0.378741
55241,Mastermind XS,Serious Problem,International,0.579918,56690,0.394900
42018,Electronischebasisgeluide,La boite a track (Spain),Electronic,0.566977,56690,0.578220


## Check whether the centre hides a taste

The first diagnostic counts candidates anchored to each seed. If one seed has almost no representation, the next step should enforce a minimum number from each seed's pool before ranking. The `closest_seed_id` is also the starting point for explaining a bridge track's connection. It is an **audio-space anchor**, not a verified perceptual explanation.

**Next step:** rank this same candidate pool into familiar, bridge and adventurous lists using centre similarity, closest-seed similarity, target distance and list diversity; compare results for the *same seed sets* against the frozen single-track baseline. Avoid selecting tiers by raw rank windows alone.

In [4]:
anchor_counts = candidates.groupby("closest_seed_id").size().reindex(SEED_IDS, fill_value=0)
display(anchor_counts.rename("candidate_count").to_frame())
# Profile and candidates are now reusable inputs to the next notebook.


,candidate_count
closest_seed_id,
704,160
1083,154
56690,190


In [5]:

print("Closest seed among the first 20 centre-ranked candidates:")
display(candidates.head(20)["closest_seed_id"].value_counts())



Closest seed among the first 20 centre-ranked candidates:


closest_seed_id
704      9
56690    9
1083     2
Name: count, dtype: int64

In [6]:
# Exclude every seed track and every track by any seed artist.
artist = small_tracks[("artist", "name")].fillna("").str.strip().str.casefold()
seed_artists = set(artist.loc[SEED_IDS])

eligible = candidates.loc[
    ~candidates.index.isin(SEED_IDS)
    & ~artist.reindex(candidates.index).isin(seed_artists)
].copy()

print(f"{len(candidates)} candidates before artist exclusion")
print(f"{len(eligible)} after artist exclusion")

for seed_id in SEED_IDS:
    print(f"\nCandidates connected to {seed_id}:")
    picks = (
        eligible[eligible["closest_seed_id"] == seed_id]
        .sort_values("closest_seed_similarity", ascending=False)
        .head(5)
    )
    display(picks.join(pd.DataFrame({
        "artist": small_tracks[("artist", "name")],
        "title": small_tracks[("track", "title")],
    }))[["artist", "title", "closest_seed_similarity", "centre_similarity"]])

504 candidates before artist exclusion
493 after artist exclusion

Candidates connected to 704:


,artist,title,closest_seed_similarity,centre_similarity
track_id,,,,
75929,Tricky Diesel,Dark Kingdom,0.633471,0.451982
141561,Yshwa,Please Listen,0.626024,0.366705
130709,Jimmy & The Threats,Silly Guy,0.607583,0.364959
36998,Exteenager,bonsoir tiroir!,0.584773,0.582421
69563,The NSJ Crew,Good Time,0.582012,0.395236



Candidates connected to 1083:


,artist,title,closest_seed_similarity,centre_similarity
track_id,,,,
131794,Pranavibes,Shanti,0.702544,0.474649
127211,Satori Ensemble,Sonar,0.637883,0.130419
5170,Peter Biedermann,Labyrinth,0.632984,0.108565
113166,I Re della Spiaggia,Nano Riflesso,0.629355,0.387918
100550,Megatone,Beyond The Universe (Transcendance),0.627074,0.316015



Candidates connected to 56690:


,artist,title,closest_seed_similarity,centre_similarity
track_id,,,,
148079,Red Star Martyrs,Into the light (Squidgy Black mix),0.694414,0.406217
114391,Quaro,Psion,0.686810,0.404841
138581,harmonic defiance,sunface,0.669833,0.479563
55233,Mastermind XS,Kingdom Come,0.665404,0.473196
22295,Comfort Fit,Sorry,0.659613,0.467610


## Multi-seed taste representation: first listening check

I tested three seed tracks that sound different from one another: 704 (Fósforo — *Mano De Dios*), 1083 (Magic Lantern — *Feasting on Energy*), and 56690 (Corty — *Trod Along*). Each seed had equal weight in the taste centre.

Retrieving candidates near both the centre and each seed produced 504 distinct tracks before excluding seed artists. Of these, 160 were closest to seed 704, 154 to 1083, and 190 to 56690. The centre represented seed 1083 less closely than the other two, so ranking only by similarity to the centre could underrepresent that part of the listener’s taste.

After excluding tracks by the seed artists, I listened to a small sample near each seed. Most sounded plausible together in a playlist. Some felt familiar; others were less familiar but retained a clear audible connection to a seed. Those less familiar tracks were convincing **bridge tracks**, although they still need enough difference to create a sense of discovery. Two recommendations near seed 704—36998 (Exteenager — *bonsoir tiroir!*) and 69563 (The NSJ Crew — *Good Time*)—felt like weak matches.

**Conclusion:** Multi-seed retrieval has produced plausible familiar and bridge candidates across different styles. These observations come from a small listening sample, so they do not measure quality across the whole candidate pool. The next step is to rank the artist-filtered candidates into Familiar, Bridge, and Adventurous lists, balancing audible connection with increasing difference, then listen to the final lists.

**Model used:** This check used the original 518-feature FMA baseline with PCA and cosine similarity. It includes MFCC but does not include the newly extracted rhythm features or percussive/harmonic ratio. The same procedure should later be run with that revised feature space.

In [7]:
from pathlib import Path

print("Working directory:", Path.cwd())

for root in [Path.cwd(), *Path.cwd().parents]:
    data_dir = root / "data"
    if data_dir.is_dir():
        print("\nData folder:", data_dir)
        for path in sorted(data_dir.rglob("*")):
            if path.is_file() and any(
                word in path.name.lower()
                for word in ("rhythm", "separation", "hpss")
            ):
                print(path)

Working directory: /Users/tim/neuefische_course/capstone-matchas-tastebridge/notebooks

Data folder: /Users/tim/neuefische_course/capstone-matchas-tastebridge/data
